# Module 2: Positional Encoding

### The concept

After Module 1, your token embeddings $z \in \mathbb{R}^{B \times N \times D}$ are position-blind — the model sees 16 vectors with no idea which patch came from where. Positional encoding fixes this by adding a unique position signal to each token:

$$z'_i = z_i + \text{PE}(i) \qquad \text{for each patch } i \in \{0, \dots, N-1\}$$

Since our patches live on a 2D grid (4×4), we use 2D sinusoidal encoding. For a patch at grid position $(r, c)$, we build its encoding by concatenating a 1D sinusoidal encoding for the row and one for the column, each of size $D/2$:

$$\text{PE}(r, c)_{2k} = \sin\!\left(\frac{r}{10000^{2k/D}}\right), \quad \text{PE}(r, c)_{2k+1} = \cos\!\left(\frac{r}{10000^{2k/D}}\right)$$

and the same pattern using $c$ for the second half of the $D$ dimensions. The result is one vector of size $D$ per patch position, giving a final encoding matrix of shape $(N, D)$.

More explicitly:

First half — encodes the row position $r$, fills dimensions $k = 0, 1, \dots, D/2 - 1$:

$$\text{PE}(r,c)_{2k} = \sin\!\left(\frac{r}{10000^{2k/(D/2)}}\right), \qquad \text{PE}(r,c)_{2k+1} = \cos\!\left(\frac{r}{10000^{2k/(D/2)}}\right)$$

Second half — encodes the column position $c$, fills dimensions $k = 0, 1, \dots, D/2 - 1$ again but placed in the upper half of the vector:

$$\text{PE}(r,c)_{D/2 + 2k} = \sin\!\left(\frac{c}{10000^{2k/(D/2)}}\right), \qquad \text{PE}(r,c)_{D/2 + 2k+1} = \cos\!\left(\frac{c}{10000^{2k/(D/2)}}\right)$$

So the full PE vector for one patch looks like:

$$\underbrace{\sin(r/\ldots),\ \cos(r/\ldots),\ \sin(r/\ldots),\ \cos(r/\ldots),\ \ldots}_{D/2 \text{ dims — row encoding}},\ \underbrace{\sin(c/\ldots),\ \cos(c/\ldots),\ \ldots}_{D/2 \text{ dims — column encoding}}$$

And $k$ is just the index stepping through the frequency levels — it runs from $0$ to $D/2 - 1$. Each successive $k$ uses a lower frequency (the denominator grows), so the sinusoids oscillate more slowly across positions.

Think of it like a hardware address bus: $k=0$ is your fastest-toggling bit, $k=D/2-1$ is your slowest. Together they uniquely identify every position.

### 🤔 Pre-coding questions

**Q1.** The encoding is added to the token, not concatenated. This means PE must be the same shape as one token embedding. Given $D=16$, what is the shape of the full positional encoding matrix for our 4×4 grid? What does each row represent?

- Answer: Token embedding are of size $z \in \mathbb{R}^{B \times N \times D}$. Each row of the PE matrix should be of size $1 \times D$. Hence, each PE matrix is of size $PE \in \mathbb{R}^{N \times D}$. Each row represents a positional encoding for that patch. Note, that the batch does not matter. This is because the positional encodings are the SAME for each batch index. The positional encodings are fixed anyway! Since the patches are fixed.

**Q2.** We split $D=16$ into two halves — 8 dimensions for the row encoding and 8 for the column encoding. Why does the model need *both* row and column components separately, rather than just encoding the flat patch $i \in \{0 \dots 15\}$?

- Answer: This is because of the 2D sinusoidal encoding. First, each row is represented with both a cos and sin operation (with the sin having an index of 2k while cos is 2k+1) so for the 4x4 case, we have 4 choices for rows but we need 8 in total so since we have both sin and cos. The same concept goes for the column. We don't combine them in one range from 0 to 15 because we want the row and column to be distinct from one another.
- We need to preserve 2D structure preservation. Consider flat index encoding: patch $i=4$ (row 1, col 0) and patch $i=1$ (row 0, col 1) would get very different flat encodings, even though they're both exactly one step away from patch $i=0$ — just in different directions. With separate row and column encodings, the model can learn "these patches share the same row" or "same column" because those axes are encoded independently. Flat encoding collapses 2D geometry into 1D and loses that structure permanently.

**Q3.** The denominator $10000^{2k/D}$ creates a range of frequencies across the $D$ dimensions. Low $k$ = high frequency (changes fast across positions), high $k$ = low frequency (changes slowly). What would go wrong if all dimensions used the same frequency?

- Answer: Each position (r,c) distinct from one another. If all frequencies are the same then there is no distinction from each position.
- Three reasons sinusoids are special:
- **Bounded**: always in $[−1,1]$ regardless of how large $r$ or $c$ gets, unlike a plain integer index which grows unboundedl
- **Relative positions are linear**: the dot product between $\text{PE}(r_1)$ and $\text{PE}(r_2)$ depends only on $r_1 - r_2$​, not on the absolute values. This means attention can learn *relative* distances naturally
- **Generalizes**: a model trained on 4×4 grids can handle larger grids at inference because the sinusoid pattern extends smoothly


**Q4.** The positional encoding is the same for every image in the batch — it's purely a function of grid position, not of pixel content. When you add it to $z \in \mathbb{R}^{B \times N \times D}$, what shape does NumPy expect the PE matrix to be, and how does broadcasting handle the batch dimension?

- Answer: Since it's just an add the numpy expects the PE matrix to be the size $\in \mathbb{R}^{N \times D}$. When numpy adds this to $\mathbb{R}^{B \times N \times D}$ it automatically adds this to each batch axis. 


**Q5.** After adding PE, two patches at different grid positions that happened to produce identical embeddings from $W_e$​ will now be distinguishable. But does the model actually *know* what row and column number a patch is at? Or just that the patches are *different from each other*?

- Answer: PE is a deterministic, fixed function of grid position. The model learns through training that certain PE patterns mean "top-left", others mean "bottom-right", and so on. What the model doesn't have is an explicit integer label saying "row=2, col=3". Instead it learns the geometry — that patches with similar row encodings are vertically aligned, patches with similar column encodings are horizontally aligned. So it knows spatial relationships implicitly, not as a named coordinate.

**Q6.** If two patches at positions $(0, 0)$ and $(0, 1)$ — same row, adjacent columns — have their PE vectors computed, which half of their $D=16$ dimensional PE vectors will be identical, and which half will be different? Why?

- Answer: First half will be the same because they represent the same rows. Second half are different because they represent the columns.

# Coding Exercise

In [5]:
# This was from mod1
import numpy as np

# ── Tiny model constants ──────────────────────────────────────
B, C, H, W  = 2, 1, 16, 16
P           = 4          # patch size
N           = (H // P) * (W // P)   # 16 patches
D           = 16         # embedding dim

# Ideally this should come from mod 1
z = np.random.randn(B, N, D) * 0.02   # (2, 16, 16)

def positional_encoding_2d(h_patches, w_patches, D):
    """
    Build a 2D sinusoidal positional encoding.

    Input:  h_patches  number of patch rows        (4)
            w_patches  number of patch columns     (4)
            D          embedding dimension          (16)
    Output: PE of shape (N, D) where N = h_patches * w_patches

    Steps:
        1. Build a row encoding  of shape (h_patches, D//2)
        2. Build a col encoding  of shape (w_patches, D//2)
        3. Expand and combine them into a (N, D) matrix
    
    Hint: np.arange, np.sin, np.cos are your friends.
          Think about what k represents and how to vectorise over it.
    """

    # --- Step 1: frequency denominators for k = 0 ... D//2-1 ---
    # denominator[k] = 10000^(2k / (D//2))
    den_k = 10000 ** (2 * np.arange(D // 2) / (D // 2))

    # --- Step 2: row encoding (h_patches, D//2) ---
    # rows shape: (h_patches, 1) so it broadcasts over k
    # note that the :: is an extended slice to fill even and odd indices separately
    # for example, 0::2 means start at index 0 and take every 2nd element (even indices)
    # 1::2 means start at index 1 and take every 2nd element (odd indices)
    rows = np.arange(h_patches)[:, None]
    row_enc = np.zeros((h_patches, D // 2))
    row_enc[:, 0::2] = np.sin(rows / den_k[0::2])
    row_enc[:, 1::2] = np.cos(rows / den_k[1::2])

    # --- Step 3: col encoding (w_patches, D//2) ---
    cols = np.arange(w_patches)[:, None]
    col_enc = np.zeros((w_patches, D // 2))
    col_enc[:, 0::2] = np.sin(cols / den_k[0::2])
    col_enc[:, 1::2] = np.cos(cols / den_k[1::2])

    # --- Step 4: expand to full grid and concatenate ---
    # Each of the h_patches rows pairs with each of the w_patches cols
    # Hint: np.repeat and np.tile, or np.meshgrid, can help here
    # Final shape before concat: both halves should be (N, D//2)
    row_enc_expanded = np.repeat(row_enc, w_patches, axis=0)
    col_enc_expanded = np.tile(col_enc, (h_patches, 1))

    # --- Step 5: concatenate row and col halves ---
    # PE shape: (N, D)
    PE = np.concatenate([row_enc_expanded, col_enc_expanded], axis=1)
    return PE


# ── Shape check ───────────────────────────────────────────────
PE = positional_encoding_2d(H // P, W // P, D)
assert PE.shape == (N, D), f"Got {PE.shape}"

# ── Add to token embeddings ───────────────────────────────────
z_pe = z + PE          # broadcasting handles the B dimension
assert z_pe.shape == (B, N, D)

print("PE:  ", PE.shape)     # expect (16, 16)
print("z_pe:", z_pe.shape)   # expect (2, 16, 16)

PE:   (16, 16)
z_pe: (2, 16, 16)


# Special Notes


### On the topic about why use sinusoidal positional encoders?
- On boundedness — you're right, for a fixed small grid it doesn't matter much. If you have a 4×4 grid, a plain integer index 0–15 is perfectly bounded and works fine. This reason matters more at scale (ViT-Large with 196 patches, or language models with thousands of tokens). For our toy model, this argument is weak.

- On relative positions being linear — this one actually matters regardless of scale, and it's worth understanding properly. When the transformer computes attention between two tokens, it takes their dot product. With sinusoidal PE, there's a elegant property: $\text{PE}(r_1) \cdot \text{PE}(r_2) = f(r_1 - r_2)$. The dot product depends only on the distance, not the absolute positions. So the attention mechanism can learn "pay attention to patches 2 steps to the right" as a single reusable pattern, regardless of where in the grid you currently are. Without this, a model trained to recognize "patch at position 3 attends to patch at position 5" would have to separately re-learn that same relationship for "patch at position 7 attends to patch at position 9." With sinusoidal PE, those two cases produce the same dot product signal — the pattern generalizes automatically. Even on a 4×4 grid, this makes training more efficient and weight-sharing more natural.

- On generalization — again, you're right that for a fixed grid size it's a non-issue. But here's a subtler point that does matter even for small grids: sinusoidal PE is not learned. It's computed analytically and frozen. That means:
No extra parameters to train
No risk of overfitting the position embeddings on small datasets
The model can in principle be applied to a different grid size at inference without retraining

If you used learned positional embeddings instead — which many modern ViTs actually do — you'd have $N \times D$ extra parameters, and they'd be meaningless for any grid size not seen during training.


### Are PEs pre-generated in practice? 
- Yes, exactly. Since sinusoidal PE is a pure function of grid position and $D$, it gets computed once at model initialization and reused for every batch forever. In PyTorch you'd typically see it registered as a buffer — not a parameter (so no gradients, no weight updates), but saved with the model. In our NumPy implementation it's the same idea: call positional_encoding_2d once and store the result.
- Some modern ViTs (like MAE, DINOv2) use learned positional embeddings instead — those start random and are trained like normal parameters. The tradeoff is exactly what we discussed: sinusoidal is free and generalizes, learned costs $N \times D$ parameters but can potentially encode more task-specific structure.

### Is D chosen to match patch size?
No — and this is an important thing to untangle. $D$ and $P^2C$ are completely independent design choices:
- $P^2 C$ is determined by your image and patch size — it's the raw input dimension $D$ is your architectural choice — how wide you want the transformer to be
- The projection matrix $W_e \in \mathbb{R}^{P^2 C \times D}$ exists precisely to bridge whatever gap exists between them. In practice they're almost never equal:
- Examples:
    - Vit-Base: $P^2 C \times D = 16^2 \times 3 = 768$, $D=768$
    - ViT-Large: $P^2 C \times D = 16^2 \times 3 = 768$, $D=1024$
    - Toy-model: $P^2 C \times D = 14^2 \times 1 = 16$, $D=16$
- Now for the PE specifically: $D$ only needs to be even so it can be split into two halves of $D/2$. Within each half, $k$ runs from $0$ to $D/2 - 1$, giving you $D/2$ frequency levels. The number of patch rows or columns has no bearing on this — you could have a 4×4 grid with $D=512$ and you'd just get 256 frequency levels per axis. More frequencies = richer position signal, but the grid size doesn't dictate it.
- The coincidence in our toy model is that $D/2 = 8$ and $h\_patches = w\_patches = 4$, which made the shapes feel like they were "matching" — but they're unrelated quantities that happened to appear together in the indexing.